# Phase 3 — Baseline points model

Train leakage-safe models to predict weekly `total_points` per player.

**Splits :**
- Train: 2016-17 → 2023-24
- Validation: 2024-25
- 2025-26 live pool is held for Phase 5 simulation (not used here)

**Models in this notebook:**
1. Rolling-form heuristic (`total_points_roll5`)
2. Cross-season anchor heuristic (`last_season_ppg`)
3. Linear regression (sklearn baseline)
4. Feedforward NN — 128 → 64 → 32, ReLU, batch norm, dropout (Phase 3 starter)

**Metrics:** MAE, RMSE, R², Spearman ρ on **all validation rows** (primary — matches FPL squad ranking). Also reported on played-only rows for reference.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "fpl_model_dataset.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

TRAIN_SEASONS = [
    "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22", "2022-23", "2023-24",
]
VAL_SEASON = "2024-25"
TARGET = "total_points"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

Project root: C:\FPL_project
Device: cpu


In [2]:
# Leakage-safe feature set 
NUMERIC_FEATURES = [
    "total_points_roll3", "total_points_roll5",
    "minutes_roll3", "minutes_roll5",
    "goals_scored_roll3", "goals_scored_roll5",
    "assists_roll3", "assists_roll5",
    "expected_goals_roll3", "expected_goals_roll5",
    "expected_assists_roll3", "expected_assists_roll5",
    "team_goals_scored_gw_roll5", "team_goals_conceded_gw_roll5", "team_points_gw_roll5",
    "opponent_team_points_roll5", "opponent_team_gc_roll5",
    "last_season_ppg", "last_season_minutes_share",
    "was_home", "rest_days",
    "value", "selected", "transfers_in", "transfers_out", "transfers_balance",
]

POSITION_MAP = {"GK": 1, "DEF": 2, "MID": 3, "FWD": 4}


def load_modeling_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["position_id"] = df["position"].map(POSITION_MAP).fillna(0)
    df["is_promoted_team"] = df["is_promoted_team"].map({True: 1.0, False: 0.0}).fillna(0.0)
    df["was_home"] = pd.to_numeric(df["was_home"], errors="coerce").fillna(0.0)
    return df


def build_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    x_num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
    x_pos = df[["position_id"]].to_numpy(dtype=np.float32)
    x_prom = df[["is_promoted_team"]].to_numpy(dtype=np.float32)
    return np.hstack([x_num, x_pos, x_prom])


def split_sets(df: pd.DataFrame):
    train = df[df["season"].isin(TRAIN_SEASONS)].copy()
    val = df[df["season"] == VAL_SEASON].copy()
    return train, val


def scored_mask(df: pd.DataFrame) -> np.ndarray:
    # Evaluate on rows where the player actually played 
    return df["minutes"].fillna(0).to_numpy() > 0


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "spearman": spearmanr(y_true, y_pred).statistic,
    }


def eval_both_slices(y_true: np.ndarray, y_pred: np.ndarray, played_mask: np.ndarray) -> dict:
    all_m = regression_metrics(y_true, y_pred)
    played_m = regression_metrics(y_true[played_mask], y_pred[played_mask])
    return {
        **all_m,
        "mae_played": played_m["mae"],
        "rmse_played": played_m["rmse"],
        "r2_played": played_m["r2"],
        "spearman_played": played_m["spearman"],
    }


df = load_modeling_table(DATA_PATH)
train_df, val_df = split_sets(df)

print(f"Rows — train: {len(train_df):,}, val: {len(val_df):,}")
print(f"Played minutes>0 — train: {scored_mask(train_df).mean():.1%}, val: {scored_mask(val_df).mean():.1%}")
print(f"Feature dim: {build_feature_matrix(train_df).shape[1]}")

Rows — train: 196,538, val: 27,605
Played minutes>0 — train: 43.6%, val: 41.9%
Feature dim: 28


## Heuristic baselines

No fitting — use features already in the v1 dataset.

In [3]:
def eval_heuristic(name: str, val: pd.DataFrame, pred_col: str) -> dict:
    mask = scored_mask(val)
    y = val[TARGET].to_numpy()
    pred = val[pred_col].fillna(0.0).to_numpy()
    out = eval_both_slices(y, pred, mask)
    out["model"] = name
    return out


baseline_rows = [
    eval_heuristic("roll5_points", val_df, "total_points_roll5"),
    eval_heuristic("last_season_ppg", val_df, "last_season_ppg"),
]
pd.DataFrame(baseline_rows)

,mae,rmse,r2,spearman,mae_played,rmse_played,r2_played,spearman_played,model
0,1.090377,2.177385,0.188338,0.675553,2.081290,3.067166,-0.122889,0.290270,roll5_points
1,1.552684,2.491475,-0.062718,0.326069,2.188108,3.063853,-0.120464,0.226227,last_season_ppg


## Linear regression baseline

Same features as the NN. Ridge regression is stable with correlated rolling stats.

In [4]:
X_train = build_feature_matrix(train_df)
y_train = train_df[TARGET].to_numpy(dtype=np.float32)
X_val = build_feature_matrix(val_df)
y_val = val_df[TARGET].to_numpy(dtype=np.float32)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

ridge = Ridge(alpha=1.0, random_state=RANDOM_SEED)
ridge.fit(X_train_s, y_train)

val_pred_ridge = ridge.predict(X_val_s)
mask = scored_mask(val_df)

ridge_metrics = eval_both_slices(y_val, val_pred_ridge, mask)
ridge_metrics["model"] = "ridge_linear"
pd.Series(ridge_metrics)

mae                    1.114626
rmse                   2.105388
r2                     0.241127
spearman               0.644458
mae_played              1.87043
rmse_played            2.840238
r2_played              0.037121
spearman_played        0.347047
model              ridge_linear
dtype: object

## Feedforward neural network (Phase 3 starter)

Architecture from `fpl_project_context.md`: 128 → 64 → 32, ReLU, batch norm, dropout.
This baseline predicts `total_points` directly (decomposed heads come in a later iteration).

In [5]:
class PointsMLP(nn.Module):
    def __init__(self, in_dim: int, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(
    X_train_s: np.ndarray,
    y_train: np.ndarray,
    X_val_s: np.ndarray,
    y_val: np.ndarray,
    val_mask: np.ndarray,
    epochs: int = 40,
    batch_size: int = 4096,
    lr: float = 1e-3,
    patience: int = 6,
):
    in_dim = X_train_s.shape[1]
    model = PointsMLP(in_dim).to(DEVICE)

    train_ds = TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_t = torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
    y_val_t = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)
    y_val_np = y_val

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    loss_fn = nn.MSELoss()

    best_state = None
    best_val_rho = -1.0
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_pred_np = val_pred.cpu().numpy()
            val_rho = spearmanr(y_val_np, val_pred_np).statistic
            val_mae = mean_absolute_error(y_val_np, val_pred_np)

        scheduler.step(val_rho)
        history.append({
            "epoch": epoch,
            "train_mse": float(np.mean(train_losses)),
            "val_spearman_all": val_rho,
            "val_mae_all": val_mae,
        })

        if val_rho > best_val_rho:
            best_val_rho = val_rho
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t).cpu().numpy()

    return model, np.array(val_pred), pd.DataFrame(history)


nn_model, val_pred_nn, train_history = train_mlp(
    X_train_s,
    y_train,
    X_val_s,
    y_val,
    scored_mask(val_df),
)

nn_metrics = eval_both_slices(y_val, val_pred_nn, scored_mask(val_df))
nn_metrics["model"] = "mlp_points"
print("Best epochs logged:", len(train_history))
pd.Series(nn_metrics)

Best epochs logged: 30


mae                  1.056206
rmse                 2.059723
r2                    0.27369
spearman             0.686446
mae_played           1.863091
rmse_played          2.799487
r2_played            0.064553
spearman_played      0.366992
model              mlp_points
dtype: object

In [6]:
comparison = pd.DataFrame(baseline_rows + [ridge_metrics, nn_metrics])
show_cols = ["model", "spearman", "mae", "spearman_played", "mae_played", "rmse", "r2"]
show_cols = [c for c in show_cols if c in comparison.columns]
comparison = comparison[show_cols].sort_values("spearman", ascending=False)
comparison

,model,spearman,mae,spearman_played,mae_played,rmse,r2
3,mlp_points,0.686446,1.056206,0.366992,1.863091,2.059723,0.273690
0,roll5_points,0.675553,1.090377,0.290270,2.081290,2.177385,0.188338
2,ridge_linear,0.644458,1.114626,0.347047,1.870430,2.105388,0.241127
1,last_season_ppg,0.326069,1.552684,0.226227,2.188108,2.491475,-0.062718


In [7]:
# Per-gameweek Spearman on validation season 
val_gw = val_df.copy()
val_gw["pred_mlp"] = val_pred_nn

gw_spearman = []
for gw, chunk in val_gw.groupby("gw"):
    if len(chunk) < 30:
        continue
    rho = spearmanr(chunk[TARGET], chunk["pred_mlp"]).statistic
    gw_spearman.append({"gw": gw, "n": len(chunk), "spearman_mlp": rho})

gw_df = pd.DataFrame(gw_spearman)
print("Mean GW Spearman (MLP):", gw_df["spearman_mlp"].mean().round(3))
gw_df.head(10)

Mean GW Spearman (MLP): 0.689


,gw,n,spearman_mlp
0,1,616,0.573738
1,2,627,0.740667
2,3,648,0.719333
3,4,659,0.680408
4,5,661,0.715797
5,6,664,0.719558
6,7,666,0.738057
7,8,667,0.727595
8,9,670,0.702015
9,10,674,0.736489


In [8]:
# Save 
val_out = val_df[["season", "element", "gw", "player_id", "name", TARGET]].copy()
val_out["pred_mlp"] = val_pred_nn
val_out["pred_ridge"] = val_pred_ridge

val_out_path = RESULTS_DIR / "val_2024_25_predictions.csv"
val_out.to_csv(val_out_path, index=False)

torch.save(
    {
        "model_state": nn_model.state_dict(),
        "in_dim": X_train_s.shape[1],
        "feature_names": NUMERIC_FEATURES + ["position_id", "is_promoted_team"],
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
    },
    RESULTS_DIR / "phase3_mlp_baseline.pt",
)

comparison.to_csv(RESULTS_DIR / "phase3_model_comparison.csv", index=False)
print("Saved:", val_out_path)
print("Saved:", RESULTS_DIR / "phase3_mlp_baseline.pt")

Saved: C:\FPL_project\results\val_2024_25_predictions.csv
Saved: C:\FPL_project\results\phase3_mlp_baseline.pt
